In [1]:
import numpy as np
import pandas as pd

In [23]:
np.random.seed(42)

n = 350

distance = np.random.uniform(1, 500, n)
order_weight = np.random.uniform(0.1, 200, n)
courier_exp = np.random.uniform(0.1, 20, n)
traffic_level = np.random.randint(1, 11, n)
weather_score = np.random.randint(1,11, n)

noise = np.random.normal(0, 0.8, n)

delivery_days = (0.5 
    + 0.015 * distance 
    + 0.05 * order_weight 
    - 0.12 * courier_exp 
    + 0.25 * traffic_level 
    + 0.20 * weather_score 
    + noise)


delivery_days = np.clip(delivery_days, 0.5, None)

df = pd.DataFrame({
    'Distance': np.round(distance, 1),
    'OrderWeight': np.round(order_weight, 1),
    'CourierExperience': np.round(courier_exp, 1),
    'TrafficLevel': traffic_level,
    'WeatherScore': weather_score,
    'DeliveryDays': np.round(delivery_days, 2)
})


print("Первые 10 записей")
print(df.head(10))


Первые 10 записей
   Distance  OrderWeight  CourierExperience  TrafficLevel  WeatherScore  \
0     187.9        100.7               10.7             7             1   
1     475.4        171.3                1.1             2             3   
2     366.3        131.8                6.8             3             9   
3     299.7         32.7                2.8             7             4   
4      78.9         14.2                1.4             4            10   
5      78.8        128.5               19.8             1             5   
6      30.0          5.4                6.5             5             2   
7     433.2        117.2               16.2             9             4   
8     301.0        188.1                5.2             8             2   
9     354.3        115.1               13.7             9             2   

   DeliveryDays  
0         10.36  
1         17.50  
2         13.75  
3          8.80  
4          5.68  
5          7.04  
6          2.52  
7         13

In [24]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Distance           350 non-null    float64
 1   OrderWeight        350 non-null    float64
 2   CourierExperience  350 non-null    float64
 3   TrafficLevel       350 non-null    int32  
 4   WeatherScore       350 non-null    int32  
 5   DeliveryDays       350 non-null    float64
dtypes: float64(4), int32(2)
memory usage: 13.8 KB


In [25]:
print(df.describe())

         Distance  OrderWeight  CourierExperience  TrafficLevel  WeatherScore  \
count  350.000000   350.000000         350.000000    350.000000    350.000000   
mean   245.388857    99.225714           9.888857      5.465714      5.505714   
std    145.555637    59.700586           5.789311      2.917730      2.904146   
min      3.500000     2.300000           0.200000      1.000000      1.000000   
25%    120.800000    46.225000           4.700000      3.000000      3.000000   
50%    255.350000    98.750000          10.000000      5.500000      5.000000   
75%    365.025000   151.050000          14.800000      8.000000      8.000000   
max    495.000000   199.900000          19.900000     10.000000     10.000000   

       DeliveryDays  
count    350.000000  
mean      10.457686  
std        4.038522  
min        0.720000  
25%        7.567500  
50%       10.385000  
75%       13.227500  
max       21.180000  


In [26]:
X = df[['Distance', 'OrderWeight', 'CourierExperience', 'TrafficLevel', 'WeatherScore']]
y = df['DeliveryDays']

In [27]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [28]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

model = LinearRegression()
model.fit(X_train, y_train)
y_pred_lr = model.predict(X_test)  

print("Linear Regression")
print("MAE:",  mean_absolute_error(y_test, y_pred_lr))
print("MSE:", mean_squared_error(y_test, y_pred_lr))
print("R²:", r2_score(y_test, y_pred_lr))

Linear Regression
MAE: 0.6779809188475645
MSE: 0.7277104900658354
R²: 0.9522999705382412


In [29]:
depths = [2, 5, 10, None]

for depth in depths:
    dt_exp = DecisionTreeRegressor(max_depth=depth, random_state=42)
    dt_exp.fit(X_train, y_train)
    
    r2_tr = r2_score(y_train, dt_exp.predict(X_train))
    r2_te = r2_score(y_test, dt_exp.predict(X_test))
    mae_te = mean_absolute_error(y_test, dt_exp.predict(X_test))
    
    print(f"max_depth={str(depth):<4} \n Train R²: {r2_tr:.4f} | Test R²: {r2_te:.4f} | Test MAE: {mae_te:.4f}")

max_depth=2    
 Train R²: 0.7042 | Test R²: 0.6904 | Test MAE: 1.8028
max_depth=5    
 Train R²: 0.9305 | Test R²: 0.8261 | Test MAE: 1.2996
max_depth=10   
 Train R²: 0.9992 | Test R²: 0.8512 | Test MAE: 1.1826
max_depth=None 
 Train R²: 1.0000 | Test R²: 0.8296 | Test MAE: 1.2709


In [30]:
dt_best = DecisionTreeRegressor(max_depth=5, random_state=42)
dt_best.fit(X_train, y_train)
y_pred_dt = dt_best.predict(X_test)

mae_dt = mean_absolute_error(y_test, y_pred_dt)
mse_dt = mean_squared_error(y_test, y_pred_dt)
r2_dt = r2_score(y_test, y_pred_dt)

results = pd.DataFrame({
    'Model': ['Linear Regression', 'Decision Tree (max_depth=5)'],
    'MAE': [round(mean_absolute_error(y_test, y_pred_lr), 3), round(mae_dt, 3)],
    'MSE': [round(mean_squared_error(y_test, y_pred_lr), 3), round(mse_dt, 3)],
    'R²': [round(r2_score(y_test, y_pred_lr), 3), round(r2_dt, 3)]
})

print(results.to_string(index=False))


                      Model   MAE   MSE    R²
          Linear Regression 0.678 0.728 0.952
Decision Tree (max_depth=5) 1.300 2.652 0.826


In [31]:
feature_cols = ['Distance', 'OrderWeight', 'CourierExperience', 'TrafficLevel', 'WeatherScore']

new_orders = pd.DataFrame([
    {'Distance': 120.0, 'OrderWeight': 5.0,  'CourierExperience': 3.0, 'TrafficLevel': 4, 'WeatherScore': 2},
    {'Distance': 450.0, 'OrderWeight': 25.0, 'CourierExperience': 10.0, 'TrafficLevel': 9, 'WeatherScore': 8},
    {'Distance': 105.0, 'OrderWeight': 1.2,  'CourierExperience': 7.5, 'TrafficLevel': 2, 'WeatherScore': 1}
])

new_orders['Predicted_DeliveryDays (LR)'] = np.round(model.predict(new_orders[feature_cols]), 2)
new_orders['Predicted_DeliveryDays (DT)'] = np.round(dt_best.predict(new_orders[feature_cols]), 2)

print("Прогноз для 3 новых заказов:")
print(new_orders.to_string(index=False))

Прогноз для 3 новых заказов:
 Distance  OrderWeight  CourierExperience  TrafficLevel  WeatherScore  Predicted_DeliveryDays (LR)  Predicted_DeliveryDays (DT)
    120.0          5.0                3.0             4             2                         3.52                         5.51
    450.0         25.0               10.0             9             8                        11.22                        10.18
    105.0          1.2                7.5             2             1                         1.90                         3.14


1) Лучший результат показала модель Linear Regression 
2) Это показывает что модель может ошибатся в прогнозе срока доставки примерно на 0,678 дня
3) Означает что линейная модель объясняет 95.2% изменчивости времени доставки на основе переданных характеристик
4) Глубина дерева — это то, насколько подробные инструкции оно себе строит. Маленькая глубина и модель слишком поверхностная и даёт частые промахи. Большая и модель пытается учесть вообще любую мелочь и начинает выдумывать правила там, где их нет
5) Потому что на тренировочных данных модель может просто зазубрить правильные ответы. На обучающих данных она покажет идеальный результат, но на настоящих новых заказах полностью провалится
6) Если на тренировочном наборе точность близка к 100%, а на тестовом ошибка резко возрастает, значит модель переобучилась и потеряла способность работать с реальными данными